In [7]:
from pathlib import Path
import pandas as pd

RAW_DATA_DIR = Path("../data/raw")

interactions_path = RAW_DATA_DIR / "Video_Games.csv.gz"
metadata_path = RAW_DATA_DIR / "meta_Video_Games.jsonl.gz"

In [8]:
interactions_sample = pd.read_csv(interactions_path, nrows=5)
interactions_sample

,user_id,parent_asin,rating,timestamp
0,AEVPPTMG43C6GWSR7I2UGRQN7WFQ,B08R5B7YS4,1.0,1611459666223
1,AEVPPTMG43C6GWSR7I2UGRQN7WFQ,B0863MT183,4.0,1613701986538
2,AEVPPTMG43C6GWSR7I2UGRQN7WFQ,B08P8P7686,5.0,1613702112995
3,AEVPPTMG43C6GWSR7I2UGRQN7WFQ,B0B7LV3DN2,4.0,1617641445475
4,AEVPPTMG43C6GWSR7I2UGRQN7WFQ,B09WMQ6DXG,5.0,1620231368468


In [9]:
metadata_sample = pd.read_json(
    metadata_path,
    lines=True,
    compression="gzip",
    nrows=5,
)

metadata_sample

,main_category,title,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together
0,Video Games,Dash 8-300 Professional Add-On,5.0,1,[Features Dash 8-300 and 8-Q300 ('Q' rollout l...,[The Dash 8-300 Professional Add-On lets you p...,NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],Aerosoft,"[Video Games, PC, Games]",{'Pricing': 'The strikethrough price is the Li...,B000FH0MHO,NaN
1,Video Games,Phantasmagoria: A Puzzle of Flesh,4.1,18,[Windows 95],[],NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],Sierra,"[Video Games, PC, Games]","{'Best Sellers Rank': {'Video Games': 137612, ...",B00069EVOG,NaN
2,Video Games,NBA 2K17 - Early Tip Off Edition - PlayStation 4,4.3,223,[The #1 rated NBA video game simulation series...,[Following the record-breaking launch of NBA 2...,58.00,[{'thumb': 'https://m.media-amazon.com/images/...,[{'title': 'NBA 2K17 - Kobe: Haters vs Players...,2K,"[Video Games, PlayStation 4, Games]","{'Release date': 'September 16, 2016', 'Best S...",B00Z9TLVK0,NaN
3,Video Games,Nintendo Selects: The Legend of Zelda Ocarina ...,4.9,22,[Authentic Nintendo Selects: The Legend of Zel...,[],37.42,[{'thumb': 'https://m.media-amazon.com/images/...,[],Amazon Renewed,"[Video Games, Legacy Systems, Nintendo Systems...","{'Best Sellers Rank': {'Video Games': 51019, '...",B07SZJZV88,NaN
4,Video Games,Thrustmaster Elite Fitness Pack for Nintendo Wii,3.0,3,"[Includes (9) Total Accessories, Pedometer, Wi...",[The Thrustmaster Motion Plus Elite Fitness Pa...,NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],THRUSTMASTER,"[Video Games, Legacy Systems, Nintendo Systems...","{'Release date': 'November 1, 2009', 'Pricing'...",B002WH4ZJG,NaN


In [10]:
print("Interaction columns:")
print(interactions_sample.dtypes)

print("\nMetadata columns:")
print(metadata_sample.dtypes)

Interaction columns:
user_id            str
parent_asin        str
rating         float64
timestamp        int64
dtype: object

Metadata columns:
main_category          str
title                  str
average_rating     float64
rating_number        int64
features            object
description         object
price              float64
images              object
videos              object
store                  str
categories          object
details             object
parent_asin            str
bought_together    float64
dtype: object


In [11]:
interactions = pd.read_csv(
    interactions_path,
    dtype={
        "user_id": "string",
        "parent_asin": "string",
        "rating": "float32",
        "timestamp": "int64",
    },
)

print("Shape:", interactions.shape)
print("\nMissing values:")
print(interactions.isna().sum())
print("\nDuplicate rows:", interactions.duplicated().sum())
print("\nRating counts:")
print(interactions["rating"].value_counts().sort_index())

Shape: (814586, 4)

Missing values:
user_id        0
parent_asin    0
rating         0
timestamp      0
dtype: int64

Duplicate rows: 0

Rating counts:
rating
1.0     52563
2.0     36809
3.0     70347
4.0    134800
5.0    520067
Name: count, dtype: int64


In [12]:
interactions["interaction_time"] = pd.to_datetime(
    interactions["timestamp"],
    unit="ms",
    utc=True,
)

print("Earliest:", interactions["interaction_time"].min())
print("Latest:", interactions["interaction_time"].max())
print("Users:", interactions["user_id"].nunique())
print("Products:", interactions["parent_asin"].nunique())

Earliest: 1999-10-22 21:43:15+00:00
Latest: 2023-09-02 18:14:28.630000+00:00
Users: 94762
Products: 25612


In [13]:
print("Shape:", interactions.shape)

Shape: (814586, 5)


In [14]:
print(interactions.isna().sum())

user_id             0
parent_asin         0
rating              0
timestamp           0
interaction_time    0
dtype: int64


In [15]:
print(interactions.duplicated().sum())

0


In [16]:
print(interactions["rating"].value_counts().sort_index())

rating
1.0     52563
2.0     36809
3.0     70347
4.0    134800
5.0    520067
Name: count, dtype: int64


In [17]:
interactions["interaction_time"] = pd.to_datetime(
    interactions["timestamp"],
    unit="ms",
    utc=True,
)

In [18]:
positive_interactions = interactions[
    interactions["rating"] >= 4
].copy()

positive_counts_per_user = (
    positive_interactions
    .groupby("user_id")
    .size()
)

eligible_user_count = (
    positive_counts_per_user >= 3
).sum()

total_user_count = interactions["user_id"].nunique()

eligible_user_percentage = (
    eligible_user_count / total_user_count * 100
)

print("Positive interactions:", len(positive_interactions))
print("Total users:", total_user_count)
print("Users with at least 3 positive interactions:", eligible_user_count)
print(f"Eligible-user percentage: {eligible_user_percentage:.2f}%")
print("\nPositive interactions per user:")
print(positive_counts_per_user.describe())

Positive interactions: 654867
Total users: 94762
Users with at least 3 positive interactions: 88137
Eligible-user percentage: 93.01%

Positive interactions per user:
count    93949.000000
mean         6.970452
std          6.624538
min          1.000000
25%          4.000000
50%          5.000000
75%          8.000000
max        328.000000
dtype: float64


### Interpretation: positive-interaction eligibility

Ratings of 4 or 5 are provisionally treated as positive interactions.

Of the 94,762 users, 88,137 have at least three positive interactions. This
means 93.01% of users can support the proposed temporal evaluation:

- Earlier positive interactions: training history
- Second-to-last positive interaction: validation target
- Last positive interaction: test target

The median user has five positive interactions, while the middle 50% have
between four and eight. Therefore, the rating threshold retains sufficient
user history for evaluation.

No records have been removed at this stage. This is a validation result, not
a completed filtering step.

The complete cumulative findings are maintained in
`reports/data_validation.md`.

In [19]:
user_product_counts = (
    interactions
    .groupby(["user_id", "parent_asin"])
    .size()
    .rename("interaction_count")
)

repeated_user_product_pairs = user_product_counts[
    user_product_counts > 1
]

print("Unique user-product pairs:", len(user_product_counts))
print("Repeated user-product pairs:", len(repeated_user_product_pairs))
print(
    "Rows belonging to repeated pairs:",
    repeated_user_product_pairs.sum()
)

if len(repeated_user_product_pairs) > 0:
    print(
        "Maximum interactions for one user-product pair:",
        repeated_user_product_pairs.max()
    )
    display(repeated_user_product_pairs.head(10))

Unique user-product pairs: 814586
Repeated user-product pairs: 0
Rows belonging to repeated pairs: 0


### Repeated user-product interactions

The dataset contains 814,586 unique user-product pairs and zero repeated
user-product pairs. Therefore, every interaction represents a distinct
user-product relationship.

No aggregation, averaging or most-recent-interaction rule is required. This
also confirms the dataset provider's stated de-duplication processing.

If repetitions had existed, their treatment would depend on whether the
system allowed repeat-product recommendations. For the current unseen-product
recommendation objective, the most recent rating would likely be retained,
while repetition count and earlier events would be preserved for analysis.

In [20]:
metadata = pd.read_json(
    metadata_path,
    lines=True,
    compression="gzip",
)

print("Metadata shape:", metadata.shape)
print("\nMetadata columns:")
print(metadata.columns.tolist())

Metadata shape: (137269, 16)

Metadata columns:
['main_category', 'title', 'average_rating', 'rating_number', 'features', 'description', 'price', 'images', 'videos', 'store', 'categories', 'details', 'parent_asin', 'bought_together', 'subtitle', 'author']


In [21]:
essential_metadata_columns = [
    "parent_asin",
    "title",
    "main_category",
    "average_rating",
    "rating_number",
]

print("Unique metadata products:", metadata["parent_asin"].nunique())

print(
    "Duplicated parent_asin values:",
    metadata["parent_asin"].duplicated().sum()
)

print("\nMissing essential metadata:")
print(metadata[essential_metadata_columns].isna().sum())

Unique metadata products: 137269
Duplicated parent_asin values: 0

Missing essential metadata:
parent_asin           0
title                 0
main_category     11035
average_rating        0
rating_number         0
dtype: int64


In [22]:
interaction_product_ids = set(
    interactions["parent_asin"].unique()
)

metadata_product_ids = set(
    metadata["parent_asin"].unique()
)

matched_product_ids = (
    interaction_product_ids & metadata_product_ids
)

missing_metadata_product_ids = (
    interaction_product_ids - metadata_product_ids
)

product_metadata_coverage = (
    len(matched_product_ids)
    / len(interaction_product_ids)
    * 100
)

interactions_without_metadata = interactions[
    interactions["parent_asin"].isin(
        missing_metadata_product_ids
    )
]

print("Products in interaction data:", len(interaction_product_ids))
print("Products matched to metadata:", len(matched_product_ids))
print(
    "Interaction products without metadata:",
    len(missing_metadata_product_ids)
)
print(f"Product metadata coverage: {product_metadata_coverage:.2f}%")
print(
    "Interaction rows without metadata:",
    len(interactions_without_metadata)
)
print(
    "Users affected by missing product metadata:",
    interactions_without_metadata["user_id"].nunique()
)

Products in interaction data: 25612
Products matched to metadata: 25612
Interaction products without metadata: 0
Product metadata coverage: 100.00%
Interaction rows without metadata: 0
Users affected by missing product metadata: 0


In [23]:
interaction_product_ids = set(
    interactions["parent_asin"].unique()
)

metadata_product_ids = set(
    metadata["parent_asin"].unique()
)

matched_product_ids = (
    interaction_product_ids & metadata_product_ids
)

missing_metadata_product_ids = (
    interaction_product_ids - metadata_product_ids
)

product_metadata_coverage = (
    len(matched_product_ids)
    / len(interaction_product_ids)
    * 100
)

interactions_without_metadata = interactions[
    interactions["parent_asin"].isin(
        missing_metadata_product_ids
    )
]

print("Products in interaction data:", len(interaction_product_ids))
print("Products matched to metadata:", len(matched_product_ids))
print(
    "Interaction products without metadata:",
    len(missing_metadata_product_ids)
)
print(f"Product metadata coverage: {product_metadata_coverage:.2f}%")
print(
    "Rating rows involving products without metadata:",
    len(interactions_without_metadata)
)
print(
    "Users affected by missing product metadata:",
    interactions_without_metadata["user_id"].nunique()
)

Products in interaction data: 25612
Products matched to metadata: 25612
Interaction products without metadata: 0
Product metadata coverage: 100.00%
Rating rows involving products without metadata: 0
Users affected by missing product metadata: 0


### Interpretation: product-metadata coverage

A metadata match means that a product's `parent_asin` appears in both the
interaction file and the metadata file. It is an exact identifier match, not
a product-similarity measurement.

All 25,612 products in the 5-core interaction data were found in the metadata
file, producing 100% product-level metadata coverage.

Therefore:

- No interaction products are missing complete metadata records.
- No rating rows are affected by absent metadata records.
- No users are affected by absent metadata records.
- No products need to be removed because an entire metadata record is missing.

Individual fields inside those metadata records may still be empty and must
be validated separately.

In [24]:
interaction_metadata = metadata[
    metadata["parent_asin"].isin(interaction_product_ids)
].copy()

print("Relevant metadata shape:", interaction_metadata.shape)

Relevant metadata shape: (25612, 16)


In [25]:
fields_to_check = [
    "title",
    "main_category",
    "average_rating",
    "rating_number",
    "price",
    "features",
    "description",
    "categories",
    "images",
]

missing_field_counts = (
    interaction_metadata[fields_to_check]
    .isna()
    .sum()
    .sort_values(ascending=False)
)

missing_field_percentages = (
    missing_field_counts
    / len(interaction_metadata)
    * 100
)

field_completeness = pd.DataFrame({
    "missing_count": missing_field_counts,
    "missing_percentage": missing_field_percentages,
})

field_completeness

,missing_count,missing_percentage
price,8477,33.097767
main_category,514,2.006872
title,0,0.000000
average_rating,0,0.000000
rating_number,0,0.000000
features,0,0.000000
description,0,0.000000
categories,0,0.000000
images,0,0.000000


In [26]:
list_fields = [
    "features",
    "description",
    "categories",
    "images",
]

empty_list_counts = {}

for field in list_fields:
    empty_list_counts[field] = (
        interaction_metadata[field]
        .apply(
            lambda value: isinstance(value, list)
            and len(value) == 0
        )
        .sum()
    )

empty_list_summary = pd.DataFrame.from_dict(
    empty_list_counts,
    orient="index",
    columns=["empty_count"],
)

empty_list_summary["empty_percentage"] = (
    empty_list_summary["empty_count"]
    / len(interaction_metadata)
    * 100
)

blank_title_count = (
    interaction_metadata["title"]
    .astype("string")
    .str.strip()
    .eq("")
    .sum()
)

print("Empty-list fields:")
display(empty_list_summary.sort_values(
    "empty_percentage",
    ascending=False,
))

print("Blank titles:", blank_title_count)

Empty-list fields:


,empty_count,empty_percentage
description,6816,26.612525
features,3294,12.861159
categories,869,3.392941
images,1,0.003904


Blank titles: 1


In [27]:
has_title = (
    interaction_metadata["title"]
    .astype("string")
    .fillna("")
    .str.strip()
    .ne("")
)

has_description = (
    interaction_metadata["description"]
    .apply(
        lambda value: isinstance(value, list)
        and any(str(item).strip() for item in value)
    )
)

has_features = (
    interaction_metadata["features"]
    .apply(
        lambda value: isinstance(value, list)
        and any(str(item).strip() for item in value)
    )
)

has_categories = (
    interaction_metadata["categories"]
    .apply(
        lambda value: isinstance(value, list)
        and any(str(item).strip() for item in value)
    )
)

has_any_text = (
    has_title
    | has_description
    | has_features
    | has_categories
)

has_rich_text = (
    has_description
    | has_features
)

print("Products with a usable title:", has_title.sum())
print("Products with a usable description:", has_description.sum())
print("Products with usable features:", has_features.sum())
print("Products with usable categories:", has_categories.sum())
print("Products with any usable text:", has_any_text.sum())
print("Products with no usable text:", (~has_any_text).sum())
print(
    "Products with neither description nor features:",
    (~has_rich_text).sum()
)

Products with a usable title: 25611
Products with a usable description: 18796
Products with usable features: 22307
Products with usable categories: 24743
Products with any usable text: 25612
Products with no usable text: 0
Products with neither description nor features: 1590


### Interpretation: metadata text availability

Metadata completeness was evaluated using both ordinary missing values and
field-specific usability checks. Empty lists and whitespace-only strings are
not classified as missing by `isna()`, but they do not provide useful text.

Among the 25,612 products used by the interaction dataset:

- 25,611 have usable titles
- 18,796 have usable descriptions
- 22,307 have usable feature text
- 24,743 have usable categories
- All 25,612 have at least one usable text source
- 1,590 have neither a usable description nor usable features

Although individual metadata fields are incomplete, every product has at
least one usable source among title, description, features and categories.

A content-based representation can therefore combine all available textual
fields and fall back to whichever fields exist for each product. No product
needs to be removed solely because it lacks all textual metadata.

Price is missing for 8,477 products (33.10%), so price should not be a
mandatory feature. Main category is missing for 514 products (2.01%), but
the more detailed `categories` list is usable for 24,743 products.

In [28]:
price_type_counts = (
    interaction_metadata["price"]
    .map(lambda value: type(value).__name__)
    .value_counts()
)

print("Price value types:")
print(price_type_counts)

price_as_numeric = pd.to_numeric(
    interaction_metadata["price"],
    errors="coerce",
)

non_numeric_price_mask = (
    interaction_metadata["price"].notna()
    & price_as_numeric.isna()
)

print(
    "\nNon-missing prices that could not be converted:",
    non_numeric_price_mask.sum()
)

print("\nExample non-numeric price values:")
display(
    interaction_metadata.loc[
        non_numeric_price_mask,
        ["parent_asin", "title", "price"],
    ].head(10)
)

Price value types:
price
float       17131
NoneType     8477
str             4
Name: count, dtype: int64

Non-missing prices that could not be converted: 4

Example non-numeric price values:


,parent_asin,title,price
30860,B0009P75HM,"Legend of Zelda: Ocarina of Time, Official Nin...",from 77.00
54198,B0146DKJGQ,Saint Seya: Soldier's Soul English/Spanish Ver...,—
91335,B00008KTX6,Super Mario Sunshine,from 64.98
101463,B000F3R79E,Alone in the Dark,from 9.99


In [29]:
interaction_metadata["price_numeric"] = pd.to_numeric(
    interaction_metadata["price"],
    errors="coerce",
)

numeric_metadata_summary = (
    interaction_metadata[
        [
            "average_rating",
            "rating_number",
            "price_numeric",
        ]
    ]
    .describe()
    .transpose()
)

invalid_average_ratings = interaction_metadata[
    interaction_metadata["average_rating"].notna()
    & ~interaction_metadata["average_rating"].between(1, 5)
]

negative_rating_counts = interaction_metadata[
    interaction_metadata["rating_number"].notna()
    & (interaction_metadata["rating_number"] < 0)
]

nonpositive_prices = interaction_metadata[
    interaction_metadata["price_numeric"].notna()
    & (interaction_metadata["price_numeric"] <= 0)
]

display(numeric_metadata_summary)

print(
    "Average ratings outside 1-5:",
    len(invalid_average_ratings)
)

print(
    "Negative rating counts:",
    len(negative_rating_counts)
)

print(
    "Non-positive numeric prices:",
    len(nonpositive_prices)
)

print(
    "Missing or non-exact numeric prices:",
    interaction_metadata["price_numeric"].isna().sum()
)

,count,mean,std,min,25%,50%,75%,max
average_rating,25612.0,4.146322,0.444798,1.0,3.90,4.20,4.5,5.00
rating_number,25612.0,825.418163,4213.211209,1.0,56.00,154.00,526.0,278574.00
price_numeric,17131.0,52.909762,86.958666,0.0,16.99,29.91,52.0,2189.75


Average ratings outside 1-5: 0
Negative rating counts: 0
Non-positive numeric prices: 6
Missing or non-exact numeric prices: 8481


In [30]:
display(
    nonpositive_prices[
        [
            "parent_asin",
            "title",
            "price",
            "price_numeric",
            "main_category",
            "categories",
        ]
    ]
)

,parent_asin,title,price,price_numeric,main_category,categories
14614,B00H8618QO,Prime World [Download],0.0,0.0,Video Games,"[Video Games, PC, Games]"
73023,B00I0E5J5Y,Path of Exile [Download],0.0,0.0,Video Games,"[Video Games, PC, Games]"
80576,B00GPQ2ZOW,Block Puzzle [Download],0.0,0.0,Video Games,"[Video Games, PC, Games]"
82159,B00HRKDECC,RAN Online [Download],0.0,0.0,Video Games,"[Video Games, PC, Games]"
82875,B00F0Q7N1I,Lord Of The Rings Online [Download],0.0,0.0,Video Games,"[Video Games, PC, Games]"
90282,B008773WWQ,Raptr [Download],0.0,0.0,Software,"[Video Games, PC, Games]"


Six downloadable PC or online products have an exact recorded price of
$0.00. Their titles and categories make a free or free-to-play interpretation
plausible, but the metadata alone does not confirm this.

A zero value could alternatively represent an unavailable price, a demo,
a product-page convention or a collection error. The original values are
preserved and flagged as ambiguous. They will not be interpreted as confirmed
free products without additional evidence.

In [31]:
earliest_interaction = interactions["interaction_time"].min()
latest_interaction = interactions["interaction_time"].max()

expected_start = pd.Timestamp(
    "1996-05-01",
    tz="UTC",
)

expected_end = pd.Timestamp(
    "2023-09-30 23:59:59.999",
    tz="UTC",
)

timestamps_outside_expected_range = interactions[
    ~interactions["interaction_time"].between(
        expected_start,
        expected_end,
    )
]

user_timestamp_counts = (
    interactions
    .groupby(["user_id", "timestamp"])
    .size()
)

tied_user_timestamps = user_timestamp_counts[
    user_timestamp_counts > 1
]

print("Earliest interaction:", earliest_interaction)
print("Latest interaction:", latest_interaction)

print(
    "Rows outside the source's expected date range:",
    len(timestamps_outside_expected_range)
)

print(
    "User-timestamp combinations containing ties:",
    len(tied_user_timestamps)
)

print(
    "Rows belonging to tied user timestamps:",
    tied_user_timestamps.sum()
)

Earliest interaction: 1999-10-22 21:43:15+00:00
Latest interaction: 2023-09-02 18:14:28.630000+00:00
Rows outside the source's expected date range: 0
User-timestamp combinations containing ties: 61
Rows belonging to tied user timestamps: 122


In [32]:
latest_positive_timestamp_per_user = (
    positive_interactions
    .groupby("user_id")["timestamp"]
    .transform("max")
)

latest_positive_rows = positive_interactions[
    positive_interactions["timestamp"]
    == latest_positive_timestamp_per_user
]

latest_positive_counts_per_user = (
    latest_positive_rows
    .groupby("user_id")
    .size()
)

users_with_tied_latest_positive = (
    latest_positive_counts_per_user[
        latest_positive_counts_per_user > 1
    ]
)

print(
    "Users with multiple positive interactions "
    "at their latest positive timestamp:",
    len(users_with_tied_latest_positive)
)

print(
    "Rows belonging to tied latest positive interactions:",
    users_with_tied_latest_positive.sum()
)

if len(users_with_tied_latest_positive) > 0:
    tied_latest_user_ids = (
        users_with_tied_latest_positive.index
    )

    display(
        latest_positive_rows[
            latest_positive_rows["user_id"].isin(
                tied_latest_user_ids
            )
        ]
        .sort_values(["user_id", "parent_asin"])
        .head(20)
    )

Users with multiple positive interactions at their latest positive timestamp: 5
Rows belonging to tied latest positive interactions: 10


,user_id,parent_asin,rating,timestamp,interaction_time
792771,AEMPKQZM62J5CJF6N3ST4MV5QWEA,B001ELJFGO,5.0,1356467054000,2012-12-25 20:24:14+00:00
792772,AEMPKQZM62J5CJF6N3ST4MV5QWEA,B002I0JBVY,5.0,1356467054000,2012-12-25 20:24:14+00:00
476986,AFFGXL2BY4A5GNTYG5Q7ICTKJUGA,B00BGA9X9W,5.0,1411246144000,2014-09-20 20:49:04+00:00
476987,AFFGXL2BY4A5GNTYG5Q7ICTKJUGA,B00CISMP8M,5.0,1411246144000,2014-09-20 20:49:04+00:00
40453,AFZISTRYDPBLS4IDSBZTKRIGKDOQ,B000NIG5Y4,5.0,1356976196000,2012-12-31 17:49:56+00:00
40452,AFZISTRYDPBLS4IDSBZTKRIGKDOQ,B000NUQQZ0,5.0,1356976196000,2012-12-31 17:49:56+00:00
333517,AH3KVQ3TR56KCZVIUUXNGR5ZCOTQ,B00EO6GSY8,5.0,1450108139000,2015-12-14 15:48:59+00:00
333518,AH3KVQ3TR56KCZVIUUXNGR5ZCOTQ,B00LSBNSAO,5.0,1450108139000,2015-12-14 15:48:59+00:00
807436,AHZ3JSFOITDDEXLQEA3WGAEKEFRA,B001ELJE1U,5.0,1483701570000,2017-01-06 11:19:30+00:00
807437,AHZ3JSFOITDDEXLQEA3WGAEKEFRA,B0084B7FZK,5.0,1483701570000,2017-01-06 11:19:30+00:00


In [33]:
tied_latest_positive_summary = (
    users_with_tied_latest_positive
    .rename("latest_positive_count")
    .to_frame()
    .join(
        positive_counts_per_user.rename(
            "total_positive_count"
        )
    )
)

tied_latest_positive_summary["eligible_for_evaluation"] = (
    tied_latest_positive_summary[
        "total_positive_count"
    ] >= 3
)

display(tied_latest_positive_summary)

,latest_positive_count,total_positive_count,eligible_for_evaluation
user_id,,,
AEMPKQZM62J5CJF6N3ST4MV5QWEA,2,7,True
AFFGXL2BY4A5GNTYG5Q7ICTKJUGA,2,8,True
AFZISTRYDPBLS4IDSBZTKRIGKDOQ,2,9,True
AH3KVQ3TR56KCZVIUUXNGR5ZCOTQ,2,5,True
AHZ3JSFOITDDEXLQEA3WGAEKEFRA,2,8,True


### Interpretation: tied latest positive timestamps

Temporal evaluation tests whether the recommender can predict later
interactions using only earlier user history.

Five users have two positive products recorded at their latest positive
timestamp. All five have at least three positive interactions and would
otherwise qualify for evaluation.

The available timestamps cannot establish which of the two products occurred
last. Selecting one based on row order or product ID would create an arbitrary
ordering.

For the primary single-target evaluation, these five users will be excluded
from the evaluated-user group. Their raw records will not be deleted. This
affects approximately 0.0057% of otherwise eligible users.

A secondary multi-target evaluation may later treat both simultaneous
products as relevant.

In [34]:
eligible_user_ids = (
    positive_counts_per_user[
        positive_counts_per_user >= 3
    ]
    .index
)

positive_timestamp_groups = (
    positive_interactions[
        positive_interactions["user_id"].isin(
            eligible_user_ids
        )
    ]
    .groupby(["user_id", "timestamp"])
    .size()
    .rename("interaction_count")
    .reset_index()
    .sort_values(
        ["user_id", "timestamp"],
        ascending=[True, False],
    )
)

positive_timestamp_groups["time_position"] = (
    positive_timestamp_groups
    .groupby("user_id")
    .cumcount()
    + 1
)

ambiguous_test_times = positive_timestamp_groups[
    (positive_timestamp_groups["time_position"] == 1)
    & (positive_timestamp_groups["interaction_count"] > 1)
]

ambiguous_validation_times = positive_timestamp_groups[
    (positive_timestamp_groups["time_position"] == 2)
    & (positive_timestamp_groups["interaction_count"] > 1)
]

print(
    "Eligible users with tied latest positive times:",
    len(ambiguous_test_times)
)

print(
    "Eligible users with tied second-latest positive times:",
    len(ambiguous_validation_times)
)

Eligible users with tied latest positive times: 5
Eligible users with tied second-latest positive times: 8


In [35]:
eligible_user_ids = (
    positive_counts_per_user[
        positive_counts_per_user >= 3
    ]
    .index
)

positive_timestamp_groups = (
    positive_interactions[
        positive_interactions["user_id"].isin(
            eligible_user_ids
        )
    ]
    .groupby(["user_id", "timestamp"])
    .size()
    .rename("interaction_count")
    .reset_index()
    .sort_values(
        ["user_id", "timestamp"],
        ascending=[True, False],
    )
)

positive_timestamp_groups["time_position"] = (
    positive_timestamp_groups
    .groupby("user_id")
    .cumcount()
    + 1
)

ambiguous_test_times = positive_timestamp_groups[
    (positive_timestamp_groups["time_position"] == 1)
    & (positive_timestamp_groups["interaction_count"] > 1)
]

ambiguous_validation_times = positive_timestamp_groups[
    (positive_timestamp_groups["time_position"] == 2)
    & (positive_timestamp_groups["interaction_count"] > 1)
]

print(
    "Eligible users with tied latest positive times:",
    len(ambiguous_test_times)
)

print(
    "Eligible users with tied second-latest positive times:",
    len(ambiguous_validation_times)
)

Eligible users with tied latest positive times: 5
Eligible users with tied second-latest positive times: 8


In [36]:
ambiguous_test_user_ids = set(
    ambiguous_test_times["user_id"]
)

ambiguous_validation_user_ids = set(
    ambiguous_validation_times["user_id"]
)

ambiguous_target_user_ids = (
    ambiguous_test_user_ids
    | ambiguous_validation_user_ids
)

users_with_both_tie_types = (
    ambiguous_test_user_ids
    & ambiguous_validation_user_ids
)

affected_user_percentage = (
    len(ambiguous_target_user_ids)
    / len(eligible_user_ids)
    * 100
)

print(
    "Users with a test-target tie:",
    len(ambiguous_test_user_ids)
)

print(
    "Users with a validation-target tie:",
    len(ambiguous_validation_user_ids)
)

print(
    "Users with both types of tie:",
    len(users_with_both_tie_types)
)

print(
    "Unique users affected by either tie:",
    len(ambiguous_target_user_ids)
)

print(
    f"Percentage of eligible users affected: "
    f"{affected_user_percentage:.4f}%"
)

Users with a test-target tie: 5
Users with a validation-target tie: 8
Users with both types of tie: 0
Unique users affected by either tie: 13
Percentage of eligible users affected: 0.0147%


In [37]:
ambiguous_test_user_ids = set(
    ambiguous_test_times["user_id"]
)

ambiguous_validation_user_ids = set(
    ambiguous_validation_times["user_id"]
)

# Users appearing in either group.
ambiguous_target_user_ids = (
    ambiguous_test_user_ids
    | ambiguous_validation_user_ids
)

# Users appearing in both groups.
users_with_both_tie_types = (
    ambiguous_test_user_ids
    & ambiguous_validation_user_ids
)

affected_user_percentage = (
    len(ambiguous_target_user_ids)
    / len(eligible_user_ids)
    * 100
)

print(
    "Users with an ambiguous test target:",
    len(ambiguous_test_user_ids),
)

print(
    "Users with an ambiguous validation target:",
    len(ambiguous_validation_user_ids),
)

print(
    "Users affected at either target position:",
    len(ambiguous_target_user_ids),
)

print(
    "Users affected at both target positions:",
    len(users_with_both_tie_types),
)

print(
    f"Percentage of eligible users affected: "
    f"{affected_user_percentage:.4f}%"
)

Users with an ambiguous test target: 5
Users with an ambiguous validation target: 8
Users affected at either target position: 13
Users affected at both target positions: 0
Percentage of eligible users affected: 0.0147%


### Interpretation: tied validation and test targets

The planned evaluation uses each eligible user's:

- latest positive-interaction time as the test-target position;
- second-latest positive-interaction time as the validation-target position;
- earlier positive interactions as training history.

A target becomes ambiguous when a user has multiple positive interactions with the same recorded timestamp at one of these target positions. The data does not provide enough information to determine which product occurred first.

The validation found:

- 5 users with ambiguous test targets;
- 8 users with ambiguous validation targets;
- 13 unique users affected across either target position;
- 0 users affected at both target positions;
- 0.0147% of eligible users affected.

For the primary single-target evaluation, these 13 users will be excluded from evaluation scoring. Their original interactions will remain unchanged in the source data.

This avoids selecting one tied product arbitrarily and incorrectly treating the other equally timed product as less relevant. Because only 0.0147% of eligible users are affected, this decision has a negligible effect on evaluation coverage while making the evaluation procedure more defensible.

A future secondary evaluation may treat all products sharing a target timestamp as a set of correct answers.

In [38]:
eligible_user_id_set = set(eligible_user_ids)

primary_evaluation_user_ids = (
    eligible_user_id_set - ambiguous_target_user_ids
)

excluded_from_primary_evaluation = (
    eligible_user_id_set & ambiguous_target_user_ids
)

primary_evaluation_coverage = (
    len(primary_evaluation_user_ids)
    / len(eligible_user_id_set)
    * 100
)

print(
    "Eligible users before resolving target ties:",
    len(eligible_user_id_set),
)

print(
    "Users excluded from primary evaluation:",
    len(excluded_from_primary_evaluation),
)

print(
    "Users retained for primary evaluation:",
    len(primary_evaluation_user_ids),
)

print(
    f"Primary evaluation coverage: "
    f"{primary_evaluation_coverage:.4f}%"
)

Eligible users before resolving target ties: 88137
Users excluded from primary evaluation: 13
Users retained for primary evaluation: 88124
Primary evaluation coverage: 99.9853%


In [39]:
positive_distinct_time_counts = (
    positive_interactions
    .groupby("user_id")["timestamp"]
    .nunique()
)

distinct_time_eligible_user_ids = set(
    positive_distinct_time_counts[
        positive_distinct_time_counts >= 3
    ].index
)

interaction_eligible_but_time_ineligible = (
    eligible_user_id_set
    - distinct_time_eligible_user_ids
)

final_primary_evaluation_user_ids = (
    primary_evaluation_user_ids
    & distinct_time_eligible_user_ids
)

final_evaluation_coverage = (
    len(final_primary_evaluation_user_ids)
    / len(eligible_user_id_set)
    * 100
)

print(
    "Users with at least 3 positive interactions:",
    len(eligible_user_id_set),
)

print(
    "Users with at least 3 distinct positive times:",
    len(distinct_time_eligible_user_ids),
)

print(
    "Interaction-eligible users with fewer than 3 distinct times:",
    len(interaction_eligible_but_time_ineligible),
)

print(
    "Users retained after all current evaluation checks:",
    len(final_primary_evaluation_user_ids),
)

print(
    f"Final evaluation-user coverage: "
    f"{final_evaluation_coverage:.4f}%"
)

Users with at least 3 positive interactions: 88137
Users with at least 3 distinct positive times: 88137
Interaction-eligible users with fewer than 3 distinct times: 0
Users retained after all current evaluation checks: 88124
Final evaluation-user coverage: 99.9853%


## Temporal-evaluation readiness decision

The proposed primary evaluation will use a time-aware leave-two-out design for positive interactions:

- ratings of 4 or 5 are provisionally treated as positive interactions;
- earlier positive interactions form the training history;
- the second-latest positive interaction becomes the validation target;
- the latest positive interaction becomes the test target.

A user must have at least three positive interactions occurring at three or more distinct timestamps.

### Validation results

- Total users: 94,762
- Users with at least three positive interactions: 88,137
- Users with at least three distinct positive-interaction times: 88,137
- Users with ambiguous test targets: 5
- Users with ambiguous validation targets: 8
- Unique users with either type of target ambiguity: 13
- Users retained for primary evaluation: 88,124
- Primary evaluation coverage: 99.9853%

### Primary evaluation policy

The 13 users with multiple positive products at a validation or test target timestamp will be excluded from primary single-target evaluation scoring.

Their original interactions will not be deleted. Multiple products sharing earlier training timestamps are allowed because every earlier interaction can remain in the user's training history.

This policy avoids inventing an order that the recorded timestamps do not provide. A future secondary multi-target evaluation may treat all positive products sharing a target timestamp as correct answers.

### Current status

This notebook validates that the proposed temporal split is feasible. It does not yet construct or save the final training, validation and test datasets. That will be performed in a separate preprocessing stage using the documented rules.